# 第 9 章 综合项目实战

**学习目标**
综合前 8 章的知识，独立完成 3 个贴近真实业务的项目：
1. **智能客服助手** —— 工具 + Agent + 多轮记忆（ch05/06）
2. **文档知识库问答** —— RAG 全流程 + 引用溯源（ch07）
3. **内容生产流水线** —— LCEL 并行 + 结构化输出（ch03/04/08）

每个项目先给**需求和验收标准**，建议自己实现后再看参考答案。

In [ ]:
# 环境准备：加载 .env 并初始化模型（本系列所有章节通用）
import os
from dotenv import load_dotenv

load_dotenv()  # 读取项目根目录下的 .env 文件

from langchain_openai import ChatOpenAI

# 智谱 GLM 提供 OpenAI 兼容接口，直接用 ChatOpenAI 指向智谱的 base_url 即可
llm = ChatOpenAI(
    model="glm-4.5-air",  # 可换成你账号可用的其他模型（改模型名即可，无需改其他代码）
    temperature=0.6,
    api_key=os.getenv("API_KEY_ZHIPU"),
    base_url=os.getenv("API_URL_ZHIPU"),  # 形如 https://open.bigmodel.cn/api/paas/v4/
)

print("模型就绪:", llm.model_name)
# 注意：本章会发起若干次真实 API 调用，会产生少量费用；练习请使用短文本。

---

## 项目 1：智能客服助手「小星」

**需求**：为一家电商公司搭建客服 Agent「小星」，能：
- 查订单状态（订单号 SO 开头）
- 算优惠价格（用户给原价和优惠方式）
- 查退换货政策（政策内容自己编 3 条）
- 记住会话中用户说过的信息（姓名、订单号）

**验收标准**
- [ ] 使用 `create_agent` + 至少 3 个工具
- [ ] 带 `checkpointer`，同 thread_id 下第二轮不用重复提供订单号
- [ ] 模型算价格必须走工具，不允许心算（写进 system_prompt 验证）
- [ ] 问"你们公司老板是谁"这类无法回答的问题时，礼貌承认不知道而不是编造

**提示**：工具里用模块级 dict 存伪数据；"不编造"靠 system_prompt 约束 + 工具返回明确的"未找到"信息。

In [ ]:
# 项目 1 参考答案
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

ORDERS = {
    "SO20260901": "已发货，运输中，预计明天送达",
    "SO20260815": "已签收",
}
POLICIES = {
    "退货": "签收后 7 天内可无理由退货，商品需完好，运费买家承担",
    "换货": "质量问题 15 天内免费换新，非质量问题不支持换货",
    "退款": "退货签收后 3 个工作日内原路退款",
}

@tool
def query_order(order_id: str) -> str:
    """根据订单号（SO 开头）查询订单物流状态。"""
    return ORDERS.get(order_id, f"未找到订单 {order_id}，请让用户核对订单号")

@tool
def calculator(expression: str) -> str:
    """计算四则运算表达式，如 '4999 * 0.9'。任何金额计算都必须用此工具。"""
    if not set(expression) <= set("0123456789+-*/(). "):
        return "错误：包含不支持的字符"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"计算出错: {e}"

@tool
def after_sale_policy(keyword: str) -> str:
    """查询售后政策，keyword 可选：退货、换货、退款。"""
    return POLICIES.get(keyword, f"暂无 '{keyword}' 相关政策")

agent = create_agent(
    model=llm,
    tools=[query_order, calculator, after_sale_policy],
    system_prompt=(
        "你是电商客服'小星'。规则：1) 涉及金额计算必须调用 calculator，禁止心算；"
        "2) 查不到的信息（如公司内部、老板是谁）要诚实说不知道；"
        "3) 记住用户在会话中提供的订单号和称呼，后续对话直接复用。"
    ),
    checkpointer=InMemorySaver(),
)
cfg = {"configurable": {"thread_id": "proj1-demo"}}

def talk(q: str):
    print("用户:", q)
    print("小星:", agent.invoke({"messages": [{"role": "user", "content": q}]}, cfg)["messages"][-1].content)
    print("-" * 50)

talk("你好，我叫老王，帮我查下订单 SO20260901")
talk("我的订单到哪了？")                                      # 复用记忆
talk("这个订单的商品原价 4999，参加满 4000 减 500 再打 9 折，帮我算下到手价")  # 强制走工具
talk("你们公司老板是谁？")                                     # 拒答测试

---

## 项目 2：文档知识库问答「内部wiki」

**需求**：把一份"新员工 FAQ"（下面给定）建成可问答的知识库：
- 切分、向量化、入库、检索、生成全流程
- 答案必须**附引用来源**（文档名）
- 知识库没有的内容明确说"无相关信息"

**验收标准**
- [ ] 用 `RecursiveCharacterTextSplitter` 切分（能说清 chunk_size 为何这么选）
- [ ] RAG 链基于 LCEL，`retriever` 以 Runnable 形式接入
- [ ] 至少 2 个"能答对"问题和 1 个"正确拒答"问题的测试记录

**提示**：参考 ch07 第 5 节的 `RunnableParallel(...).assign(answer=...)` 模式。

In [ ]:
# 项目 2 参考答案
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# 向量模型（同 ch07 的配置）
embeddings = OpenAIEmbeddings(
    model="embedding-3", dimensions=1024,
    api_key=os.getenv("API_KEY_ZHIPU"), base_url=os.getenv("API_URL_ZHIPU"),
)

wiki_text = (
    "新员工入职 FAQ（人力资源部）\n\n"
    "Q: 试用期多久？A: 试用期 3 个月，表现优秀可提前转正，需直属上级和 HR 双重审批。\n\n"
    "Q: 工资哪天发？A: 每月 10 日发放上月工资，遇节假日提前至最近工作日。\n\n"
    "Q: 有加班费吗？A: 公司实行调休制，工作日加班按 1:1 调休，周末加班 1:1.5，需提前在 OA 报备。\n\n"
    "Q: 社保公积金基数？A: 按实际工资全额缴纳，公积金比例 12%。\n\n"
    "Q: 电脑设备怎么领？A: 入职当天到 IT 服务台领取，笔记本型号可自选 Mac 或 ThinkPad。"
)

docs = [Document(page_content=wiki_text, metadata={"source": "新员工FAQ.md"})]

splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20,
                                          separators=["\n\n", "\n", "。", " ", ""])
chunks = splitter.split_documents(docs)
print(f"切分出 {len(chunks)} 块")

store = InMemoryVectorStore(embedding=embeddings)
store.add_documents(chunks)
retriever = store.as_retriever(search_kwargs={"k": 2})

def format_docs(docs):
    return "\n\n".join(f"【{d.metadata['source']}】{d.page_content}" for d in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "仅根据资料回答，并注明来源文档名；资料中没有就回答'知识库中无相关信息'。\n\n资料:\n{context}"),
    ("human", "{question}"),
])

qa = RunnableParallel(docs=retriever, question=RunnablePassthrough()).assign(
    answer=(
        {"context": lambda x: format_docs(x["docs"]), "question": lambda x: x["question"]}
        | prompt | llm | StrOutputParser()
    )
)

def ask(q: str):
    r = qa.invoke(q)
    print(f"Q: {q}\nA: {r['answer']}")
    print("命中块:", [d.page_content[:30] + "..." for d in r["docs"]])
    print("-" * 50)

ask("试用期可以提前转正吗？")        # ✅ 能答
ask("加班怎么算？")                  # ✅ 能答
ask("公司有健身房吗？")              # ✅ 正确拒答

---

## 项目 3：内容生产流水线「一键成稿」

**需求**：输入一个产品名，产出一份结构化的营销包：
- 并行生成：标题、卖点（3 条）、短文案（50 字内）
- 用**结构化输出**把结果定形为 Pydantic 模型（不允许自由文本）
- 最后用代码把结构化结果渲染成 Markdown 模板

**验收标准**
- [ ] 三个生成任务**并行**执行（`RunnableParallel`）
- [ ] 生成阶段用 `with_structured_output` 保证 schema
- [ ] 渲染阶段是**纯代码**，不调用 LLM

**提示**：先 `RunnableParallel` 拿三路输出，再接一个 `RunnableLambda` 做代码渲染；结构化输出放在每条分支链的解析器位置。

In [ ]:
# 项目 3 参考答案
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda

class Title(BaseModel):
    headline: str = Field(description="产品标题，10 字以内")

class SellingPoints(BaseModel):
    points: list[str] = Field(description="3 条卖点，每条不超过 12 字")

class Copy(BaseModel):
    text: str = Field(description="50 字以内的种草短文案")

def branch(schema, system):
    return ChatPromptTemplate.from_messages([("system", system), ("human", "{product}")]) \
        | llm.with_structured_output(schema, method="function_calling")  # 智谱需显式指定 function_calling

def render_markdown(d: dict) -> str:
    points = "\n".join(f"- {p}" for p in d["selling"].points)
    return (
        f"# {d['title'].headline}\n\n"
        f"## 核心卖点\n{points}\n\n"
        f"## 种草文案\n> {d['copy'].text}\n"
    )

pipeline = RunnableParallel(
    title=branch(Title, "你是标题专家，为产品起一个吸引人的标题"),
    selling=branch(SellingPoints, "你是营销专家，提炼产品的 3 条核心卖点"),
    copy=branch(Copy, "你是种草文案高手，写一段口吻像朋友推荐的短文案"),
) | RunnableLambda(render_markdown)   # 最后一步是纯代码渲染，不调用 LLM

print(pipeline.invoke({"product": "便携咖啡机"}))

---

## 下一步学习路线

完成本章后，你已经具备 LangChain 的核心能力。建议按这个顺序进阶：

| 方向 | 学什么 | 为什么 |
|---|---|---|
| **LangGraph** | StateGraph、条件边、human-in-the-loop | 复杂多步工作流、多 Agent 协作的事实标准（`create_agent` 内部就是它） |
| **LangSmith** | 追踪、评估、数据集 | 生产环境调试与质量回归的必备可观测平台 |
| **RAG 进阶** | 混合检索、重排序（rerank）、Parent-Document、多轮 query 改写 | 把 ch07 的玩具 RAG 变成能打的 RAG |
| **MCP** | Model Context Protocol | 工具生态标准协议，让 Agent 接入现成的外部工具 |
| **部署** | LangServe / FastAPI + SSE | 把 Notebook 里的链变成线上服务 |

> 一个检验学习成果的好办法：把本章 3 个项目**重构成 FastAPI 接口**（流式接口用 SSE），你就同时打通了开发和服务端两块拼图。